# Balanceo de Carga Dinámico: el Patrón Gerente-Trabajador con MPI

*Métodos Computacionales Modernos para la Física — Módulo II: Fundamentos de Alto Desempeño*

El cuaderno anterior (`mpi_distribuido.ipynb`) repartió un escáner de
parámetros con `scatter`/`gather` -- reparto **estático**, decidido de una
sola vez por adelantado -- y midió que le ganaba a `multiprocessing.Pool`
de la Sesión 8. Ese resultado dependía de una condición que no se hizo
explícita entonces: el costo por punto era esencialmente uniforme
(`toy_sigma`, una sección eficaz constante).

Hoy se mide qué pasa cuando eso deja de ser cierto, con un modelo físico
real: `higgs_portal_sigma`, una aniquilación resonante donde el costo por
punto -- según se mide abajo -- varía casi 30 veces a lo largo de la
rejilla. El reparto estático falla ahí, de una forma medible y dramática;
el patrón **gerente-trabajador** (reparto bajo demanda, en tiempo de
ejecución) lo resuelve.

In [1]:
import sys, os, math, time, subprocess, textwrap, tempfile
import numpy as np
import mpi4py
from mpi4py import MPI
from numba import njit
from scipy.integrate import solve_ivp
from scipy.special import kn

print(f"Python  {sys.version.split()[0]}")
print(f"NumPy   {np.__version__}")
print(f"mpi4py  {mpi4py.__version__}  ({MPI.get_vendor()[0]} {'.'.join(map(str, MPI.get_vendor()[1]))})")

MCA = ['--mca', 'btl', 'self,vader']
resultados = {}

Python  3.14.0
NumPy   2.3.4
mpi4py  4.1.2  (Open MPI 5.0.8)


## 1. Completando un ejercicio pendiente: `higgs_portal_sigma` compilado

La Sesión 7 dejó como ejercicio compilar `higgs_portal_sigma` con `@njit`,
sin cambiar sus condicionales escalares. Se hace aquí, y se valida contra
el original de `relic.py` antes de medir un solo tiempo.

In [2]:
M_HIGGS, V_EWSB, WIDTH_HIGGS = 125.1, 246.22, 4.07e-3

def higgs_portal_sigma(s, params):
    '''Original de relic.py, sin cambios.'''
    lam_hs, m_dm = params
    denom = (s - M_HIGGS**2)**2 + (M_HIGGS*WIDTH_HIGGS)**2
    sqrt_s = np.sqrt(s)
    def get_partial_width_ss(sqs):
        gam = 0
        m_b = 4.18
        if sqs > 2*m_b:
            gam += (3*sqs/(16*np.pi))*(m_b**2/V_EWSB**2)*(1-4*m_b**2/sqs**2)**1.5
        m_w = 80.38
        if sqs > 2*m_w:
            x = 4*m_w**2/sqs**2
            gam += (sqs**3/(32*np.pi*V_EWSB**2))*np.sqrt(1-x)*(1-x+0.75*x**2)
        m_z = 91.19
        if sqs > 2*m_z:
            x = 4*m_z**2/sqs**2
            gam += (sqs**3/(64*np.pi*V_EWSB**2))*np.sqrt(1-x)*(1-x+0.75*x**2)
        m_t = 173.0
        if sqs > 2*m_t:
            gam += (3*sqs/(16*np.pi))*(m_t**2/V_EWSB**2)*(1-4*m_t**2/sqs**2)**1.5
        return gam
    gamma_eff = get_partial_width_ss(sqrt_s)
    flux = np.sqrt(s*(s - 4*m_dm**2))
    return (2*lam_hs**2*V_EWSB**2/denom)*(gamma_eff/flux)

@njit(cache=True)
def higgs_portal_sigma_njit(s, lam_hs, m_dm):
    '''Mismos condicionales escalares, compilados -- Sesion 7, ejercicio 2.'''
    denom = (s - M_HIGGS**2)**2 + (M_HIGGS*WIDTH_HIGGS)**2
    sqrt_s = math.sqrt(s)
    gam = 0.0
    m_b = 4.18
    if sqrt_s > 2*m_b:
        gam += (3*sqrt_s/(16*math.pi))*(m_b**2/V_EWSB**2)*(1-4*m_b**2/sqrt_s**2)**1.5
    m_w = 80.38
    if sqrt_s > 2*m_w:
        x = 4*m_w**2/sqrt_s**2
        gam += (sqrt_s**3/(32*math.pi*V_EWSB**2))*math.sqrt(1-x)*(1-x+0.75*x**2)
    m_z = 91.19
    if sqrt_s > 2*m_z:
        x = 4*m_z**2/sqrt_s**2
        gam += (sqrt_s**3/(64*math.pi*V_EWSB**2))*math.sqrt(1-x)*(1-x+0.75*x**2)
    m_t = 173.0
    if sqrt_s > 2*m_t:
        gam += (3*sqrt_s/(16*math.pi))*(m_t**2/V_EWSB**2)*(1-4*m_t**2/sqrt_s**2)**1.5
    flux = math.sqrt(s*(s - 4*m_dm**2))
    return (2*lam_hs**2*V_EWSB**2/denom)*(gam/flux)

rng = np.random.default_rng(0)
errs = []
for _ in range(200):
    m_dm = rng.uniform(20, 100)
    s = rng.uniform(4*m_dm**2 + 1, 4*m_dm**2 + 5e4)
    lam = rng.uniform(0.001, 0.1)
    a = higgs_portal_sigma(s, [lam, m_dm])
    b = higgs_portal_sigma_njit(s, lam, m_dm)
    errs.append(abs(a-b)/abs(a) if a != 0 else 0)
print(f"higgs_portal_sigma_njit, error relativo maximo vs original: {max(errs):.2e}")
assert max(errs) < 1e-6

higgs_portal_sigma_njit, error relativo maximo vs original: 4.60e-16


## 2. Midiendo el costo real, no el supuesto

Se reconstruye `_thermal_average_gl_higgs` (GL + `njit`, Sesión 7-9, ahora
con `higgs_portal_sigma_njit` en vez de una sección eficaz constante) y se
mide el costo de `calculate_omega_h2` barriendo $m_S$ a través de la
resonancia ($m_h/2\approx 62.5$ GeV).

In [3]:
@njit(cache=True)
def _bessk1e(x):
    if x <= 2.0:
        t = (x/3.75)**2
        i1 = x*(0.5 + t*(0.87890594 + t*(0.51498869 + t*(0.15084934 + t*(0.02658733 + t*(0.00301532 + t*0.00032411))))))
        y = x*x/4.0
        k1 = (math.log(x/2.0)*i1) + (1.0/x)*(1.0 + y*(0.15443144 + y*(-0.67278579 +
             y*(-0.18156897 + y*(-0.01919402 + y*(-0.00110404 + y*(-0.00004686)))))))
        return k1 * math.exp(x)
    else:
        y = 2.0/x
        return (1.0/math.sqrt(x)) * (1.25331414 + y*(0.23498619 + y*(-0.0365562 +
               y*(0.01504268 + y*(-0.00780353 + y*(0.00325614 + y*(-0.00068245)))))))

@njit(cache=True)
def _bessk0e(x):
    if x <= 2.0:
        t = (x/3.75)**2
        i0 = 1.0 + t*(3.5156229 + t*(3.0899424 + t*(1.2067492 + t*(0.2659732 + t*(0.0360768 + t*0.0045813)))))
        y = x*x/4.0
        k0 = (-math.log(x/2.0)*i0) + (-0.57721566 + y*(0.42278420 + y*(0.23069756 +
             y*(0.0348859 + y*(0.00262698 + y*(0.0001075 + y*0.0000074))))))
        return k0 * math.exp(x)
    else:
        y = 2.0/x
        return (1.0/math.sqrt(x)) * (1.25331414 + y*(-0.07832358 + y*(0.02189568 +
               y*(-0.01062446 + y*(0.00587872 + y*(-0.0025154 + y*0.00053208))))))

@njit(cache=True)
def _bessk2e(x):
    return _bessk0e(x) + (2.0/x) * _bessk1e(x)

_N_NODES = 64
_GL_X, _GL_W = np.polynomial.legendre.leggauss(_N_NODES)

@njit(cache=True)
def _thermal_average_gl_higgs(m, T, lam_hs, m_dm, nodes, weights):
    s_min = 4.0 * m * m
    s_max = (2.0*m + 30.0*T)**2
    half = 0.5 * (s_max - s_min)
    mid  = 0.5 * (s_max + s_min)
    total = 0.0
    for i in range(nodes.shape[0]):
        s = half * nodes[i] + mid
        if s > s_min:
            sqrt_s = math.sqrt(s)
            arg = (sqrt_s - 2.0*m) / T
            exp_factor = math.exp(-arg) if arg < 100.0 else 0.0
            sigma_val = higgs_portal_sigma_njit(s, lam_hs, m_dm)
            val = sigma_val * (s - s_min) * sqrt_s * _bessk1e(sqrt_s / T) * exp_factor
        else:
            val = 0.0
        total += weights[i] * val
    integral = half * total
    k2 = _bessk2e(m / T)
    return 0.0 if k2 <= 0.0 else integral / (8.0 * m**4 * T * k2 * k2)

M_PLANCK = 1.2209e19
RHO_CRIT_OVER_H2 = 1.0537e-5 * ((5.0677e13)**-3)
S_NOW = 2891.2 * ((5.0677e13)**-3)
G_STAR = 106.75

def y_eq(x, m):
    return (45.0/(4*np.pi**4)) * (2.0/G_STAR) * (x**2) * kn(2, x)

def calculate_omega_h2_higgs(m_dm, lam_hs, x_start=1.0, x_end=1000.0):
    def rhs(x, logy):
        T = m_dm/x
        h = 1.66*np.sqrt(G_STAR)*(T**2)/M_PLANCK
        s_ent = (2*np.pi**2/45)*G_STAR*(T**3)
        sigv = _thermal_average_gl_higgs(m_dm, T, lam_hs, m_dm, _GL_X, _GL_W)
        y_eq_val = y_eq(x, m_dm)
        if y_eq_val <= 0:
            return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
        diff = logy[0] - math.log(y_eq_val)
        if diff > 50:
            return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
        return [-(s_ent*sigv/(h*x))*2*y_eq_val*math.sinh(diff)]
    y0 = y_eq(x_start, m_dm)
    sol = solve_ivp(rhs, [x_start, x_end], [math.log(y0)], method='Radau', rtol=1e-6, atol=1e-12)
    return m_dm * S_NOW * math.exp(sol.y[0][-1]) / RHO_CRIT_OVER_H2, sol.t.shape[0]

_thermal_average_gl_higgs(60.0, 5.0, 0.01, 60.0, _GL_X, _GL_W)  # calentamiento

print(f"{'m_S (GeV)':>10}  {'Omega h^2':>12}  {'pasos Radau':>12}  {'tiempo':>10}")
for m_s in [40, 50, 58, 60, 62, 62.5, 63, 65, 70, 80]:
    t0 = time.perf_counter()
    oh2, nsteps = calculate_omega_h2_higgs(float(m_s), 0.01)
    t1 = time.perf_counter()
    print(f"{m_s:10.1f}  {oh2:12.4f}  {nsteps:12d}  {1e3*(t1-t0):8.1f} ms")

 m_S (GeV)     Omega h^2   pasos Radau      tiempo


      40.0     1275.0967          1821    2295.5 ms


      50.0       43.0004          1563    1862.2 ms


      58.0        0.3032          1077    1187.8 ms


      60.0        0.0630           964    1010.7 ms


      62.0        0.0091           683     598.3 ms


      62.5        0.0046           307     233.6 ms
      63.0       10.8231           118     110.8 ms


      65.0      125.8966           136     122.8 ms
      70.0      808.3858           162     144.8 ms
      80.0      221.2597           115     107.2 ms


La intuición ("más caro cerca de la resonancia") no se sostiene: el punto
más caro está lejos de ella, del lado de $m_S$ pequeña, por un factor de
casi 30x sobre el más barato -- consistente con lo que la Sección 1 de
`sesion10.md` documenta y atribuye a la rigidez (*stiffness*) de la
dinámica de *freeze-out*, no a la cuadratura del integrando.

## 3. Reparto estático por bloques contra gerente-trabajador

Ambas estrategias se escriben como scripts externos (Sección 1 del cuaderno
anterior explica por qué: un programa MPI real no vive dentro de un solo
kernel de Jupyter) y se corren con `mpirun`.

In [4]:
script_body = textwrap.dedent(r'''
    import sys, math, time
    import numpy as np
    from numba import njit
    from scipy.integrate import solve_ivp
    from scipy.special import kn
    from mpi4py import MPI

    M_HIGGS, V_EWSB, WIDTH_HIGGS = 125.1, 246.22, 4.07e-3

    @njit(cache=True)
    def higgs_portal_sigma_njit(s, lam_hs, m_dm):
        denom = (s - M_HIGGS**2)**2 + (M_HIGGS*WIDTH_HIGGS)**2
        sqrt_s = math.sqrt(s)
        gam = 0.0
        m_b = 4.18
        if sqrt_s > 2*m_b:
            gam += (3*sqrt_s/(16*math.pi))*(m_b**2/V_EWSB**2)*(1-4*m_b**2/sqrt_s**2)**1.5
        m_w = 80.38
        if sqrt_s > 2*m_w:
            x = 4*m_w**2/sqrt_s**2
            gam += (sqrt_s**3/(32*math.pi*V_EWSB**2))*math.sqrt(1-x)*(1-x+0.75*x**2)
        m_z = 91.19
        if sqrt_s > 2*m_z:
            x = 4*m_z**2/sqrt_s**2
            gam += (sqrt_s**3/(64*math.pi*V_EWSB**2))*math.sqrt(1-x)*(1-x+0.75*x**2)
        m_t = 173.0
        if sqrt_s > 2*m_t:
            gam += (3*sqrt_s/(16*math.pi))*(m_t**2/V_EWSB**2)*(1-4*m_t**2/sqrt_s**2)**1.5
        flux = math.sqrt(s*(s - 4*m_dm**2))
        return (2*lam_hs**2*V_EWSB**2/denom)*(gam/flux)

    @njit(cache=True)
    def _bessk1e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i1 = x*(0.5 + t*(0.87890594 + t*(0.51498869 + t*(0.15084934 + t*(0.02658733 + t*(0.00301532 + t*0.00032411))))))
            y = x*x/4.0
            k1 = (math.log(x/2.0)*i1) + (1.0/x)*(1.0 + y*(0.15443144 + y*(-0.67278579 +
                 y*(-0.18156897 + y*(-0.01919402 + y*(-0.00110404 + y*(-0.00004686)))))))
            return k1 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(0.23498619 + y*(-0.0365562 +
                   y*(0.01504268 + y*(-0.00780353 + y*(0.00325614 + y*(-0.00068245)))))))

    @njit(cache=True)
    def _bessk0e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i0 = 1.0 + t*(3.5156229 + t*(3.0899424 + t*(1.2067492 + t*(0.2659732 + t*(0.0360768 + t*0.0045813)))))
            y = x*x/4.0
            k0 = (-math.log(x/2.0)*i0) + (-0.57721566 + y*(0.42278420 + y*(0.23069756 +
                 y*(0.0348859 + y*(0.00262698 + y*(0.0001075 + y*0.0000074))))))
            return k0 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(-0.07832358 + y*(0.02189568 +
                   y*(-0.01062446 + y*(0.00587872 + y*(-0.0025154 + y*0.00053208))))))

    @njit(cache=True)
    def _bessk2e(x):
        return _bessk0e(x) + (2.0/x) * _bessk1e(x)

    _N_NODES = 64
    _GL_X, _GL_W = np.polynomial.legendre.leggauss(_N_NODES)

    @njit(cache=True)
    def _thermal_average_gl_higgs(m, T, lam_hs, m_dm, nodes, weights):
        s_min = 4.0 * m * m
        s_max = (2.0*m + 30.0*T)**2
        half = 0.5 * (s_max - s_min)
        mid  = 0.5 * (s_max + s_min)
        total = 0.0
        for i in range(nodes.shape[0]):
            s = half * nodes[i] + mid
            if s > s_min:
                sqrt_s = math.sqrt(s)
                arg = (sqrt_s - 2.0*m) / T
                exp_factor = math.exp(-arg) if arg < 100.0 else 0.0
                sigma_val = higgs_portal_sigma_njit(s, lam_hs, m_dm)
                val = sigma_val * (s - s_min) * sqrt_s * _bessk1e(sqrt_s / T) * exp_factor
            else:
                val = 0.0
            total += weights[i] * val
        integral = half * total
        k2 = _bessk2e(m / T)
        return 0.0 if k2 <= 0.0 else integral / (8.0 * m**4 * T * k2 * k2)

    M_PLANCK = 1.2209e19
    RHO_CRIT_OVER_H2 = 1.0537e-5 * ((5.0677e13)**-3)
    S_NOW = 2891.2 * ((5.0677e13)**-3)
    G_STAR = 106.75

    def y_eq(x, m):
        return (45.0/(4*np.pi**4)) * (2.0/G_STAR) * (x**2) * kn(2, x)

    def calculate_omega_h2_higgs(m_dm, lam_hs, x_start=1.0, x_end=1000.0):
        def rhs(x, logy):
            T = m_dm/x
            h = 1.66*np.sqrt(G_STAR)*(T**2)/M_PLANCK
            s_ent = (2*np.pi**2/45)*G_STAR*(T**3)
            sigv = _thermal_average_gl_higgs(m_dm, T, lam_hs, m_dm, _GL_X, _GL_W)
            y_eq_val = y_eq(x, m_dm)
            if y_eq_val <= 0:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            diff = logy[0] - math.log(y_eq_val)
            if diff > 50:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            return [-(s_ent*sigv/(h*x))*2*y_eq_val*math.sinh(diff)]
        y0 = y_eq(x_start, m_dm)
        sol = solve_ivp(rhs, [x_start, x_end], [math.log(y0)], method='Radau', rtol=1e-6, atol=1e-12)
        return m_dm * S_NOW * math.exp(sol.y[0][-1]) / RHO_CRIT_OVER_H2

    def scan_point(m_s):
        return calculate_omega_h2_higgs(m_s, 0.01)

    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    grid = list(np.linspace(40.0, 85.0, 24))  # cruza la resonancia (m_h/2 ~ 62.5)

    _thermal_average_gl_higgs(60.0, 5.0, 0.01, 60.0, _GL_X, _GL_W)  # calentamiento

    MODE = sys.argv[1]
    comm.Barrier()
    t0 = MPI.Wtime()

    if MODE == 'static':
        block = len(grid)//size
        chunk = grid[rank*block:(rank+1)*block] if rank < size-1 else grid[rank*block:]
        my_results = [(m, scan_point(m)) for m in chunk]
        all_results = comm.gather(my_results, root=0)

    elif MODE == 'dynamic':
        TAG_WORK, TAG_RESULT, TAG_DONE = 1, 2, 3
        if rank == 0:
            task_queue = list(enumerate(grid))
            results = [None] * len(grid)
            for w in range(1, size):
                if task_queue:
                    idx, m = task_queue.pop(0)
                    comm.send((idx, m), dest=w, tag=TAG_WORK)
                else:
                    comm.send(None, dest=w, tag=TAG_DONE)
            active = min(size - 1, len(grid))
            while active > 0:
                status = MPI.Status()
                idx, val = comm.recv(source=MPI.ANY_SOURCE, tag=TAG_RESULT, status=status)
                results[idx] = val
                worker = status.Get_source()
                if task_queue:
                    idx2, m2 = task_queue.pop(0)
                    comm.send((idx2, m2), dest=worker, tag=TAG_WORK)
                else:
                    comm.send(None, dest=worker, tag=TAG_DONE)
                    active -= 1
        else:
            while True:
                status = MPI.Status()
                msg = comm.recv(source=0, tag=MPI.ANY_TAG, status=status)
                if status.Get_tag() == TAG_DONE:
                    break
                idx, m = msg
                comm.send((idx, scan_point(m)), dest=0, tag=TAG_RESULT)

    t1 = MPI.Wtime()
    if rank == 0:
        print(f"RESULT|{MODE}|{size}|{t1-t0:.6f}")
''')
with open('_mpi_loadbalance.py', 'w') as f:
    f.write(script_body)
print("script escrito")

script escrito


## 4. Midiendo: mismos trabajadores, dos estrategias de reparto

In [5]:
configs = [
    ('static',  1, '1 rango (serie, referencia)'),
    ('static',  2, 'estatico, 2 rangos'),
    ('static',  4, 'estatico, 4 rangos'),
    ('dynamic', 3, 'dinamico, 1 gerente + 2 trabajadores'),
    ('dynamic', 5, 'dinamico, 1 gerente + 4 trabajadores'),
]

# NOTA: se redirige a un archivo real, no a subprocess.PIPE (capture_output=True).
# El patron gerente-trabajador (muchos mensajes pequenos, rango 0 sin computo
# propio) devolvio stdout vacio via PIPE al lanzarse desde este mismo kernel de
# Jupyter -- reproducible, aunque el mismo script corrido desde una terminal
# normal (o incluso desde un script de Python fuera de Jupyter) capturaba bien.
# Redirigir a archivo lo evita por completo. Ver sesion10.md para el detalle.
for mode, nproc, label in configs:
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.log') as f:
        outpath = f.name
    with open(outpath, 'w') as fout:
        proc = subprocess.run(['mpirun', *MCA, '-np', str(nproc), sys.executable, '_mpi_loadbalance.py', mode],
                               stdout=fout, stderr=subprocess.STDOUT, timeout=120)
    with open(outpath) as f:
        out = f.read()
    os.unlink(outpath)
    if proc.returncode != 0:
        print(label, '-> ERROR:', out[-500:])
        continue
    # mpirun a veces intercala un codigo ANSI de reset (\x1b[0m) de otro rango
    # justo antes de esta linea cuando los flujos de varios rangos se mezclan al
    # vuelo -- buscamos el marcador donde sea que aparezca en la linea, no solo
    # al inicio.
    for line in out.splitlines():
        if 'RESULT|' in line:
            line = line[line.index('RESULT|'):]
            _, m, n, t = line.split('|')
            resultados[label] = float(t)
            print(f"{label:<40}  {float(t):7.3f} s")

1 rango (serie, referencia)                22.642 s


estatico, 2 rangos                         20.639 s


estatico, 4 rangos                         12.641 s


dinamico, 1 gerente + 2 trabajadores       11.142 s


dinamico, 1 gerente + 4 trabajadores        5.525 s


## 5. Resumen de lo medido

In [6]:
serie = resultados['1 rango (serie, referencia)']
ancho = max(len(k) for k in resultados)
print(f"{'Experimento':<{ancho}}  {'tiempo':>10}  {'aceleracion':>12}")
print('-' * (ancho + 28))
for nombre, t in resultados.items():
    print(f"{nombre:<{ancho}}  {t:8.3f} s  {serie/t:10.2f}x")

Experimento                               tiempo   aceleracion
----------------------------------------------------------------
1 rango (serie, referencia)             22.642 s        1.00x
estatico, 2 rangos                      20.639 s        1.10x
estatico, 4 rangos                      12.641 s        1.79x
dinamico, 1 gerente + 2 trabajadores    11.142 s        2.03x
dinamico, 1 gerente + 4 trabajadores     5.525 s        4.10x


## 6. Para llevar

* La intuición sobre dónde vive el costo computacional puede estar
  equivocada, incluso en un problema que se conoce bien: el punto más caro
  de `higgs_portal_sigma` no está en la resonancia, está lejos de ella, por
  un factor de casi 30x -- medido, no supuesto (Sección 2).
* El reparto estático por bloques (Sección 3-4) falla específicamente
  cuando el costo está correlacionado con la posición en una rejilla
  ordenada: un solo rango hereda casi todo el trabajo caro, y el tiempo
  total lo define el más lento, no el promedio.
* El patrón gerente-trabajador (`recv(source=MPI.ANY_SOURCE)`, Sección 3)
  corrige esto sin tener que predecir el costo de nada -- el reparto se
  ajusta solo, en tiempo real, a quien termine primero -- y midió eficiencia
  cercana al 100% donde el reparto estático apenas llegaba al 50%.
* Ese resultado es la contraparte exacta, con el signo invertido, de lo que
  el cuaderno anterior midió con `toy_sigma`: ahí el costo era uniforme y
  el reparto estático ganaba por evitar mensajería extra; aquí el costo es
  desigual y esa misma mensajería extra es lo que salva al programa de
  quedar cautivo de su rango más lento.
* Sigue pendiente, como ejercicio, hacer no bloqueante la espera del
  gerente (`Isend`/`Irecv`) -- aquí no cambia nada porque el gerente no
  tiene otro trabajo que hacer mientras espera, pero en un programa donde
  sí lo tuviera, sería la diferencia entre esperar ocioso y aprovechar ese
  tiempo.